#  Europe Under Pressure — Part 4
## The Housing Affordability Shock Index (HASI)

**Series:** Europe Under Pressure | **Author:** Poonum

**Theoretical Backbone:** Knoll, Schularick & Steger (2017), *'No Price Like Home: Global House Prices, 1870–2012'*, American Economic Review, 107(2), 331–353.

**Supporting Evidence:** IMF Working Paper (2023), *'European Housing Markets at a Turning Point'*

---

### Motivation

Knoll et al. (2017) showed that real house prices across advanced economies stayed flat for a century, then surged after WWII — with **rising land prices (not construction costs) explaining ~80% of the global housing boom**. Today, that boom has become a structural shock for European households.

This notebook constructs a **Housing Affordability Shock Index (HASI)** for 16 EU countries and tests whether housing stress predicts unemployment — extending our series-wide investigation into Europe's structural vulnerabilities.

---

In [9]:
# ── Install dependencies ────────────────────────────────────────────
!pip install plotly statsmodels kaleido -q

In [10]:
# ── Imports ─────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import statsmodels.api as sm
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

print(' All libraries loaded')

 All libraries loaded


## Section 1: Data

| Variable | Source | Code | Year |
|---|---|---|---|
| Price-to-Income Ratio | OECD Analytical House Price Database | HM1.2.1 | 2023 |
| Housing Cost Overburden Rate | Eurostat EU-SILC | `ilc_lvho07a` | 2023 |
| Unemployment Rate | Eurostat | `une_rt_a` | 2023 |

**Price-to-Income Ratio:** Nominal house prices ÷ nominal disposable income per head, base 2015=100. Value of 150 = housing 50% less affordable than 2015.

**Overburden Rate:** % of population spending >40% of income on housing (EU's official stress threshold).

In [11]:
# ── Data ─────────────────────────────────────────────────────────────
# 16 EU countries — same sample as Parts 1–3

COUNTRIES = [
    'Austria', 'Belgium', 'Czech Republic', 'Denmark', 'Finland',
    'France', 'Germany', 'Greece', 'Hungary', 'Ireland',
    'Italy', 'Netherlands', 'Poland', 'Portugal', 'Spain', 'Sweden'
]

ISO3 = {
    'Austria':'AUT','Belgium':'BEL','Czech Republic':'CZE','Denmark':'DNK',
    'Finland':'FIN','France':'FRA','Germany':'DEU','Greece':'GRC',
    'Hungary':'HUN','Ireland':'IRL','Italy':'ITA','Netherlands':'NLD',
    'Poland':'POL','Portugal':'PRT','Spain':'ESP','Sweden':'SWE'
}

# OECD Price-to-Income Ratio 2023 (base 2015=100)
# Source: OECD Analytical House Price Database, HM1.2.1
price_to_income = {
    'Austria':120.1, 'Belgium':118.4, 'Czech Republic':148.2, 'Denmark':118.9,
    'Finland':91.3,  'France':113.6,  'Germany':138.7,        'Greece':109.4,
    'Hungary':181.3, 'Ireland':137.5, 'Italy':94.2,           'Netherlands':140.6,
    'Poland':152.4,  'Portugal':150.8,'Spain':118.2,          'Sweden':111.6
}

# Eurostat Housing Cost Overburden Rate 2023 (%)
# Source: Eurostat EU-SILC, ilc_lvho07a, total population
overburden = {
    'Austria':6.2,  'Belgium':7.8,  'Czech Republic':8.4,  'Denmark':15.1,
    'Finland':6.1,  'France':5.6,   'Germany':12.2,        'Greece':33.8,
    'Hungary':7.8,  'Ireland':5.4,  'Italy':5.5,           'Netherlands':3.8,
    'Poland':6.2,   'Portugal':6.0, 'Spain':8.1,           'Sweden':10.3
}

# Eurostat Unemployment Rate 2023 (%)
# Source: Eurostat, une_rt_a, annual average
unemployment = {
    'Austria':5.0,  'Belgium':5.5,  'Czech Republic':2.6,  'Denmark':5.1,
    'Finland':7.5,  'France':7.3,   'Germany':3.0,         'Greece':11.1,
    'Hungary':4.1,  'Ireland':4.3,  'Italy':6.7,           'Netherlands':3.6,
    'Poland':2.9,   'Portugal':6.5, 'Spain':12.2,          'Sweden':8.5
}

df = pd.DataFrame({
    'Country':             COUNTRIES,
    'ISO3':                [ISO3[c] for c in COUNTRIES],
    'PriceToIncome_2023':  [price_to_income[c] for c in COUNTRIES],
    'OverburdenRate_2023': [overburden[c] for c in COUNTRIES],
    'Unemployment_2023':   [unemployment[c] for c in COUNTRIES],
})

print(f' Dataset loaded: {len(df)} countries')
df

 Dataset loaded: 16 countries


,Country,ISO3,PriceToIncome_2023,OverburdenRate_2023,Unemployment_2023
0,Austria,AUT,120.1,6.2,5.0
1,Belgium,BEL,118.4,7.8,5.5
2,Czech Republic,CZE,148.2,8.4,2.6
3,Denmark,DNK,118.9,15.1,5.1
4,Finland,FIN,91.3,6.1,7.5
5,France,FRA,113.6,5.6,7.3
6,Germany,DEU,138.7,12.2,3.0
7,Greece,GRC,109.4,33.8,11.1
8,Hungary,HUN,181.3,7.8,4.1
9,Ireland,IRL,137.5,5.4,4.3


## Section 2: Constructing the HASI

$$\text{HASI}_i = \frac{z(\text{PriceToIncome}_i) + z(\text{OverburdenRate}_i)}{2}$$

Equal-weight composite of both z-scored components. Higher HASI = more severely housing-shocked.

In [12]:
# ── Build HASI ───────────────────────────────────────────────────────
df['z_PTI']        = (df.PriceToIncome_2023  - df.PriceToIncome_2023.mean())  / df.PriceToIncome_2023.std()
df['z_Overburden'] = (df.OverburdenRate_2023 - df.OverburdenRate_2023.mean()) / df.OverburdenRate_2023.std()
df['HASI']         = (df.z_PTI + df.z_Overburden) / 2

df = df.sort_values('HASI', ascending=False).reset_index(drop=True)

print('=== HASI Rankings ===')
print(df[['Country','PriceToIncome_2023','OverburdenRate_2023','HASI']].to_string(index=False))

=== HASI Rankings ===
       Country  PriceToIncome_2023  OverburdenRate_2023      HASI
        Greece               109.4                 33.8  1.328738
       Hungary               181.3                  7.8  1.028840
       Germany               138.7                 12.2  0.435524
Czech Republic               148.2                  8.4  0.370350
        Poland               152.4                  6.2  0.305104
      Portugal               150.8                  6.0  0.257230
       Denmark               118.9                 15.1  0.219655
       Ireland               137.5                  5.4 -0.066287
   Netherlands               140.6                  3.8 -0.112777
        Sweden               111.6                 10.3 -0.271135
         Spain               118.2                  8.1 -0.285587
       Belgium               118.4                  7.8 -0.302372
       Austria               120.1                  6.2 -0.378491
        France               113.6                  5.

## Section 3: Visualisation 1 — HASI Bar Chart

In [13]:
colors = ['#e8712a' if v > 0 else '#2e6da4' for v in df['HASI']]

fig1 = go.Figure(go.Bar(
    x=df['HASI'], y=df['Country'],
    orientation='h',
    marker_color=colors,
    text=[f"{v:+.2f}" for v in df['HASI']],
    textposition='outside',
    textfont=dict(size=11)
))
fig1.add_vline(x=0, line_dash='dash', line_color='grey', line_width=1)
fig1.update_layout(
    title=dict(
        text='<b>Housing Affordability Shock Index (HASI) — EU, 2023</b><br>'
             '<sup>Composite of OECD Price-to-Income Ratio & Eurostat Overburden Rate | '
             'Theory: Knoll, Schularick & Steger (2017)</sup>',
        x=0.5, font=dict(size=15, color='#1a3a5c')
    ),
    xaxis_title='HASI Score (higher = more housing-stressed)',
    yaxis=dict(autorange='reversed', tickfont=dict(size=12)),
    plot_bgcolor='#f4f6f9', paper_bgcolor='white',
    height=560, width=820,
    margin=dict(l=150, r=80, t=110, b=60),
    annotations=[dict(
        text='Source: OECD Analytical House Price Database (HM1.2.1); Eurostat EU-SILC (ilc_lvho07a)',
        x=0, y=-0.1, xref='paper', yref='paper',
        showarrow=False, font=dict(size=9, color='grey')
    )]
)
fig1.show()
fig1.write_html('fig1_HASI_bar.html', include_plotlyjs='cdn')
print(' fig1_HASI_bar.html saved')

 fig1_HASI_bar.html saved


## Section 4: Visualisation 2 — Choropleth Map

In [14]:
fig2 = go.Figure(go.Choropleth(
    locations=df['ISO3'],
    z=df['HASI'],
    colorscale=[[0,'#c6dbef'],[0.5,'#4292c6'],[1,'#08306b']],
    zmin=df['HASI'].min(), zmax=df['HASI'].max(),
    colorbar=dict(title='HASI', thickness=15, len=0.6),
    marker_line_color='white', marker_line_width=0.5,
    text=df['Country'],
    hovertemplate='<b>%{text}</b><br>HASI: %{z:.2f}<extra></extra>'
))
fig2.update_layout(
    title=dict(
        text='<b>Housing Affordability Shock Across the EU, 2023</b><br>'
             '<sup>Darker = More Housing-Stressed</sup>',
        x=0.5, font=dict(size=15, color='#1a3a5c')
    ),
    geo=dict(
        scope='europe', projection_type='natural earth',
        showland=True, landcolor='#f0f0f0',
        showcoastlines=True, coastlinecolor='white',
        showframe=False,
        center=dict(lat=54, lon=15), projection_scale=2.2
    ),
    height=520, width=820,
    paper_bgcolor='white',
    annotations=[dict(
        text='Source: OECD & Eurostat | Theory: Knoll, Schularick & Steger (2017)',
        x=0.5, y=-0.04, xref='paper', yref='paper',
        showarrow=False, font=dict(size=9, color='grey')
    )]
)
fig2.show()
fig2.write_html('fig2_HASI_map.html', include_plotlyjs='cdn')
print(' fig2_HASI_map.html saved')

 fig2_HASI_map.html saved


## Section 5: OLS Regression — Does Housing Shock Predict Unemployment?

$$\text{Unemployment}_i = \alpha + \beta \cdot \text{HASI}_i + \varepsilon_i$$

**Hypothesis:** Housing unaffordability creates a *labour mobility trap* — workers can't relocate to where jobs are because housing is too expensive everywhere — predicting **higher** unemployment. This would break the series-wide pattern.

In [15]:
# ── OLS Regression ───────────────────────────────────────────────────
X = sm.add_constant(df['HASI'])
y = df['Unemployment_2023']
model = sm.OLS(y, X).fit()
print(model.summary())

b  = model.params['HASI']
p  = model.pvalues['HASI']
r2 = model.rsquared
print(f'\nβ = {b:.3f} | p = {p:.3f} | R² = {r2:.3f}')

                            OLS Regression Results                            
Dep. Variable:      Unemployment_2023   R-squared:                       0.014
Model:                            OLS   Adj. R-squared:                 -0.056
Method:                 Least Squares   F-statistic:                    0.2010
Date:                Fri, 05 Jun 2026   Prob (F-statistic):              0.661
Time:                        10:04:22   Log-Likelihood:                -38.700
No. Observations:                  16   AIC:                             81.40
Df Residuals:                      14   BIC:                             82.94
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          5.9938      0.726      8.252      0.0

## Section 6: Visualisation 3 — Scatter + OLS

In [16]:
x_line = np.linspace(df['HASI'].min() - 0.1, df['HASI'].max() + 0.1, 100)
y_line = model.params['const'] + model.params['HASI'] * x_line

fig3 = go.Figure()
fig3.add_trace(go.Scatter(
    x=x_line, y=y_line, mode='lines',
    line=dict(color='#e8712a', width=2, dash='dash'),
    name=f'OLS: β={b:.2f}, p={p:.2f}, R²={r2:.2f}'
))
fig3.add_trace(go.Scatter(
    x=df['HASI'], y=df['Unemployment_2023'],
    mode='markers+text', text=df['Country'],
    textposition='top center', textfont=dict(size=9),
    marker=dict(size=10, color='#2e6da4', line=dict(color='#1a3a5c', width=1)),
    name='EU Countries'
))
fig3.update_layout(
    title=dict(
        text='<b>HASI vs Unemployment — The Labour Mobility Trap Test</b><br>'
             '<sup>OLS Regression: Housing Affordability Shock → Unemployment Rate (2023)</sup>',
        x=0.5, font=dict(size=15, color='#1a3a5c')
    ),
    xaxis_title='HASI Score',
    yaxis_title='Unemployment Rate 2023 (%)',
    plot_bgcolor='#f4f6f9', paper_bgcolor='white',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.8)', bordercolor='#ccc', borderwidth=1),
    height=520, width=820,
    margin=dict(l=60, r=40, t=110, b=80),
    annotations=[dict(
        text='Source: OECD, Eurostat | Unemployment: Eurostat une_rt_a',
        x=0.5, y=-0.14, xref='paper', yref='paper',
        showarrow=False, font=dict(size=9, color='grey')
    )]
)
fig3.show()
fig3.write_html('fig3_HASI_scatter.html', include_plotlyjs='cdn')
print(' fig3_HASI_scatter.html saved')

 fig3_HASI_scatter.html saved


## Section 7: Discussion

### HASI Rankings

**Greece** tops the index — not from price growth (PTI = 109.4) but from a devastating overburden rate of **33.8%**, nearly triple the EU average. Wages collapsed during the sovereign debt crisis and never recovered, making even modestly priced housing unaffordable.

**Hungary** ranks second with PTI = **181.3** — the highest in our sample. Budapest saw extraordinary house price inflation driven by speculative demand and tax incentives for homeownership far outpacing wage growth.

**Czech Republic, Poland, Portugal** cluster with PTI above 148 — Central and Southern European markets where wage growth significantly lagged house prices through the 2020s.

**Italy and Finland** score lowest — Italy because its housing market stagnated after 2008; Finland because its welfare system provides substantial housing support.

### The Regression — Series Pattern Holds

**β = –0.53, p = 0.66, R² = 0.014** — the coefficient is *negative* again, consistent with Parts 1, 2, and 3.

The labour mobility trap hypothesis does not hold statistically. Two possible explanations:
1. **Labour market dualism:** Housing-stressed economies (Germany, Netherlands) have flexible labour markets and strong export sectors sustaining employment despite cost pressures.
2. **Reverse causality:** Tight housing markets often coincide with strong job markets attracting workers in the first place.

### Series-Wide Finding

Across all four shocks — China trade competition, AI automation risk, CBAM climate exposure, and housing unaffordability — a **consistent negative (or null) β** emerges. Europe's labour markets appear structurally resilient to shock exposure in ways that simple theory does not predict.

---

## References

- Knoll, K., Schularick, M., & Steger, T. (2017). No Price Like Home: Global House Prices, 1870–2012. *American Economic Review*, 107(2), 331–353.
- IMF (2023). European Housing Markets at a Turning Point. *IMF Working Paper* WP/23/76.
- OECD (2024). OECD Analytical House Price Database. https://stats.oecd.org
- Eurostat (2024). Housing cost overburden rate (ilc_lvho07a). EU-SILC.
- Eurostat (2024). Unemployment rate (une_rt_a).